# Fine-tuning FreeCAD Model with Mistral Studio

This notebook demonstrates how to fine-tune a Mistral model for FreeCAD design tasks using Mistral Studio.

## Setup and Configuration

First, let's configure the Mistral Studio API access.

In [ ]:
# Install required packages
%pip install mistralai datasets python-dotenv

# Import libraries
import os
from dotenv import load_dotenv
from mistralai import Mistral

# Load environment variables
load_dotenv()

# Configure Mistral Studio API
MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY')
client = Mistral(api_key=MISTRAL_API_KEY)

## Load FreeCAD Dataset

Load the FreeCAD dataset we created earlier.

In [ ]:
import json

# Load FreeCAD dataset
with open('../../finetune/datasets/freecad_chat.jsonl', 'r') as f:
    dataset = [json.loads(line) for line in f]

print(f'Loaded {len(dataset)} FreeCAD examples')
print('Sample entry:')
print(json.dumps(dataset[0], indent=2))

## Fine-tuning with Mistral Studio

Submit the dataset to Mistral Studio for fine-tuning.

In [ ]:
# Prepare dataset for Mistral Studio format
mistral_dataset = []
for entry in dataset:
    messages = entry['messages']
    # Convert to Mistral Studio format
    mistral_entry = {
        'messages': messages
    }
    mistral_dataset.append(mistral_entry)

print(f'Prepared {len(mistral_dataset)} entries for Mistral Studio')

In [ ]:
# Submit fine-tuning job to Mistral Studio
# Note: This is a conceptual example - actual API may differ

fine_tune_response = client.fine_tuning.create(
    model='mistral-large-latest',
    training_data=mistral_dataset,
    hyperparameters={
        'epochs': 3,
        'learning_rate': 2e-5,
        'batch_size': 16
    },
    name='freecad-designer-v1'
)

print('Fine-tuning job submitted:')
print(f'Job ID: {fine_tune_response.id}')
print(f'Status: {fine_tune_response.status}')

## Monitor Fine-tuning Progress

Check the status of your fine-tuning job.

In [ ]:
# Check job status
job_status = client.fine_tuning.get(fine_tune_response.id)

print(f'Job Status: {job_status.status}')
print(f'Progress: {job_status.progress}%')
print(f'Estimated Completion: {job_status.estimated_completion}')

## Test the Fine-tuned Model

Once fine-tuning is complete, test the model.

In [ ]:
# Use the fine-tuned model
def test_freecad_model(model_name='ft:freecad-designer-v1'):
    test_prompt = '''Explain how to create a parametric gear in FreeCAD
    with proper constraints and expressions.'''
    
    response = client.chat.complete(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a FreeCAD expert."},
            {"role": "user", "content": test_prompt}
        ]
    )
    
    return response.choices[0].message.content

# Test when model is ready
# freecad_explanation = test_freecad_model()
# print('FreeCAD Gear Creation Guide:')
# print(freecad_explanation)

## Integration with Mascarade

Once you have your fine-tuned model, integrate it with Mascarade.

In [ ]:
# Update Mascarade configuration
mascarade_config = {
    'mistral_api_key': MISTRAL_API_KEY,
    'mistral_default_model': 'ft:freecad-designer-v1',
    'agents': [
        {
            'name': 'freecad-designer',
            'description': 'FreeCAD design expert with Mistral Studio fine-tuning',
            'preferred_provider': 'mistral',
            'preferred_model': 'ft:freecad-designer-v1'
        }
    ]
}

print('Configuration for Mascarade integration:')
print(json.dumps(mascarade_config, indent=2))